# 03 · Inference demo: the committed VBX models on held-out patients

Everything below is computed when the notebook runs, from files committed in this repository:
`artifacts/models/*` (the deployed model sets, sha256-checked against `artifacts/manifest.json`) and
`artifacts/features/features.parquet` (radiomics-style features of the UCSF-PDGM cohort). The API uses the
same loading and prediction code (`backend/app/domain`), so these are the outputs the app shows.

Run from the repository root or from `ml/notebooks/` with the backend environment
(`backend/requirements-dev.txt`).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT=Path(".").resolve() if Path("artifacts").exists() else Path("../..").resolve()
sys.path[:0]=[str(ROOT/"backend"), str(ROOT/"ml"/"pipeline")]

import sklearn, lightgbm
from app.domain.contracts import Marker, MARKER_LABELS
from app.domain.registry import load_registry
from app.domain.feature_store import load_feature_store
from app.domain.profile import build_profile, known_label
from app.core.logging import configure_logging

configure_logging("WARNING")  # keep the output to results; the API logs model loading at INFO

registry=load_registry(ROOT/"artifacts", ROOT/"ml"/"configs"/"markers.yaml")
store=load_feature_store(ROOT/"artifacts"/"features"/"features.parquet")
table=pd.read_parquet(ROOT/"artifacts"/"features"/"features.parquet")
test=table[table["split"]=="test"]
print("scikit-learn", sklearn.__version__, "| lightgbm", lightgbm.__version__)
print("patients:", len(table), "| held-out test patients:", len(test))
print("model sets loaded:", sum(s=="loaded" for s in registry.status.values()), "of", len(registry.status))

scikit-learn

1.9.1

| lightgbm

4.7.0

patients:

...

99

model sets loaded:

15

of

15

## 1. The deployed model sets

Each marker has a near-optimal set of equally good models (cross-validated log-loss within ε of the best).
`skill` is the improvement of the best model over always predicting the training base rate; below 5% the set is
flagged low-signal and the app shows "No reliable signal" instead of a verdict.

In [2]:
rows=[]
for (marker, variant), p in sorted(registry.predictors.items(), key=lambda kv: (kv[0][0].value, kv[0][1])):
    m=p.meta
    rows.append({
        "marker": marker.value, "variant": variant, "models": len(m.models),
        "families": ", ".join(sorted({x.family for x in m.models})),
        "best CV log-loss": round(min(x.cv_log_loss for x in m.models), 4),
        "base-rate log-loss": m.baseline_log_loss,
        "skill": None if m.skill is None else f"{m.skill:.1%}",
        "low signal": p.low_signal,
    })
pd.DataFrame(rows)

,marker,variant,models,families,best CV log-loss,base-rate log-loss,skill,low signal
0,1p19q,clinical_only,10,logreg,0.1615,0.1691,4.5%,True
1,1p19q,fused,5,"extratrees, logreg",0.1359,0.1691,19.6%,False
2,1p19q,fused_given_idh,6,"extratrees, lightgbm, logreg",0.1281,0.1691,24.2%,False
3,1p19q,imaging_only,5,extratrees,0.1357,0.1691,19.8%,False
4,GRADE,clinical_only,10,logreg,0.3944,0.4969,20.6%,False
5,GRADE,fused,5,"extratrees, lightgbm, logreg",0.1251,0.4969,74.8%,False
6,GRADE,fused_given_idh,5,logreg,0.0995,0.4969,80.0%,False
7,GRADE,imaging_only,5,"lightgbm, logreg",0.1239,0.4969,75.1%,False
8,IDH,clinical_only,11,logreg,0.3050,0.5053,39.6%,False
9,IDH,fused,5,"lightgbm, logreg",0.1993,0.5053,60.6%,False


## 2. Inference on sample inputs

Three held-out patients (never used in training). For each marker: every model's probability, the set's
median, how many models agree, the stability verdict and the call, next to the patient's true lab result.

In [3]:
SAMPLES=["UCSF-PDGM-0162", "UCSF-PDGM-0243", "UCSF-PDGM-0061"]
LABEL_COLUMN={Marker.IDH: "idh", Marker.CODEL_1P19Q: "codeletion_1p19q", Marker.MGMT: "mgmt", Marker.GRADE: "who_grade"}

def vector(patient_id, known=None):
    row=store.rows[patient_id]
    biopsy=row.get("prior_biopsy")
    return store.vector_for(
        patient_id=patient_id, study_id=patient_id, age=int(round(row["age"])),
        sex="M" if int(row["sex"])==1 else "F",
        prior_biopsy=None if biopsy is None or biopsy!=biopsy else bool(biopsy), known_markers=known or {},
    )

def truth(patient_id, marker):
    value=store.rows[patient_id].get(LABEL_COLUMN[marker])
    if value is None or value!=value:
        return "not recorded"
    return known_label(marker, int(value==4) if marker==Marker.GRADE else int(value))

out=[]
for pid in SAMPLES:
    for r in build_profile(registry, vector(pid)):
        p=r.prediction
        out.append({
            "patient": pid, "age": store.rows[pid]["age"], "marker": r.marker.value,
            "model probabilities": " ".join(f"{x:.2f}" for x in p.model_probabilities),
            "median": round(p.prob_median, 3), "agree": f"{round(p.agreement*p.n_models)}/{p.n_models}",
            "verdict": "NO RELIABLE SIGNAL" if r.low_signal else p.verdict.value,
            "call": p.call, "true result": truth(pid, r.marker),
            "resolved by combining": r.resolved_by_combining,
        })
pd.DataFrame(out)

,patient,age,marker,model probabilities,median,agree,verdict,call,true result,resolved by combining
0,UCSF-PDGM-0162,33.0,IDH,0.98 0.99 0.88 0.99 0.89,0.981,5/5,STABLE,mutant,mutant,False
1,UCSF-PDGM-0162,33.0,1p19q,0.03 0.04 0.03 0.08 0.04,0.037,5/5,STABLE,intact,intact,False
2,UCSF-PDGM-0162,33.0,MGMT,0.79 0.75 0.77 0.74 0.74 0.74 0.77 0.74 0.76 0...,0.756,24/24,NO RELIABLE SIGNAL,methylated,methylated,False
3,UCSF-PDGM-0162,33.0,GRADE,0.93 0.53 0.94 0.54 0.93,0.926,5/5,CONTESTED,grade 4,grade 4,False
4,UCSF-PDGM-0243,59.0,IDH,0.61 0.60 0.47 0.59 0.52,0.587,4/5,CONTESTED,mutant,mutant,False
5,UCSF-PDGM-0243,59.0,1p19q,0.05 0.05 0.05 0.09 0.04,0.046,5/5,STABLE,intact,codeleted,False
6,UCSF-PDGM-0243,59.0,MGMT,0.75 0.76 0.74 0.76 0.76 0.75 0.75 0.76 0.74 0...,0.746,24/24,NO RELIABLE SIGNAL,methylated,not recorded,False
7,UCSF-PDGM-0243,59.0,GRADE,0.75 0.46 0.81 0.44 0.84,0.755,3/5,ARBITRARY,grade 4,grade 2-3,False
8,UCSF-PDGM-0061,46.0,IDH,0.22 0.12 0.35 0.06 0.35,0.216,5/5,CONTESTED,wildtype,mutant,False
9,UCSF-PDGM-0061,46.0,1p19q,0.03 0.03 0.03 0.03 0.03,0.030,5/5,STABLE,intact,intact,False


## 3. Entering a known lab result

When IDH is already known from tissue, it becomes an input: the other markers are re-scored by the
`fused_given_idh` sets, which were trained with the IDH label as a feature.

In [4]:
pid="UCSF-PDGM-0243"
before={r.marker: r for r in build_profile(registry, vector(pid))}
after={r.marker: r for r in build_profile(registry, vector(pid, {Marker.IDH: 1}))}
rows=[]
for marker in (Marker.CODEL_1P19Q, Marker.MGMT, Marker.GRADE):
    b, a=before[marker], after[marker]
    rows.append({
        "marker": marker.value,
        "before (set, median, verdict, call)": f"{b.variant}, {b.prediction.prob_median:.2f}, {b.prediction.verdict.value}, {b.call}",
        "with IDH = mutant": f"{a.variant}, {a.prediction.prob_median:.2f}, {a.prediction.verdict.value}, {a.call}",
        "true result": truth(pid, marker),
    })
print(pid, "true IDH:", truth(pid, Marker.IDH))
pd.DataFrame(rows)

UCSF-PDGM-0243

true IDH:

mutant

,marker,"before (set, median, verdict, call)",with IDH = mutant,true result
0,1p19q,"fused, 0.05, STABLE, intact","fused_given_idh, 0.12, STABLE, intact",codeleted
1,MGMT,"fused, 0.75, STABLE, methylated","fused_given_idh, 0.77, STABLE, methylated",not recorded
2,GRADE,"fused, 0.75, ARBITRARY, grade 4","fused_given_idh, 0.37, STABLE, grade 2-3",grade 2-3


## 4. Why a call was made, and what would flip it

Grouped perturbation: each input group is reset to the training-cohort median and every model is re-run.
Tipping points sweep clinician-verifiable inputs over plausible values. Both describe model sensitivity, not cause.

In [5]:
from app.domain.explain import explain_set
from app.domain.tipping import find_tipping_points

pid="UCSF-PDGM-0162"
idh=registry.for_patient(Marker.IDH, set())
explanation=explain_set(idh, vector(pid))
labels={e.group: e.label for e in explanation.effects}
print(pid, "IDH call:", idh.predict_set(vector(pid)).call,
      "| models agree on why (mean rank correlation):", round(explanation.explanation_agreement, 2))
display(pd.DataFrame([
    {"input group": labels[i.group], "pushes toward": "mutant" if i.direction=="toward_positive" else "wildtype",
     "mean effect (probability points)": round(i.weight*100, 1), "models agreeing": i.agreement}
    for i in explanation.items[:6]
]))
for t in find_tipping_points(idh, vector(pid), store.descriptor_grids, MARKER_LABELS[Marker.IDH]):
    print(f"{t.label}: now {t.current:.3g} -> " + (f"flips to {t.flips_to} at {t.flip_at:.3g}" if t.flip_at is not None else "no plausible value flips the call"))

UCSF-PDGM-0162

IDH call:

mutant

| models agree on why (mean rank correlation):

0.81

...

Age: now 33 -> no plausible value flips the call

Tumor volume: now 1.2e+05 -> no plausible value flips the call

Enhancing share of the tumor: now 0.0811 -> no plausible value flips the call

Necrotic share of the tumor: now 0.0136 -> no plausible value flips the call

Edema-to-core ratio: now 9.56 -> no plausible value flips the call

## 5. Doubt: why a call is uncertain

Patient 0061's IDH call is wrong (true result: mutant). The doubt decomposition flags the tumor as unlike the
training patients (nearest-neighbour distance in imaging-feature space), which is the cue not to rely on it.

In [6]:
from app.domain.doubt import decompose, primary

pid="UCSF-PDGM-0061"
pred=registry.for_patient(Marker.IDH, set())
prediction=pred.predict_set(vector(pid))
components=decompose(prediction, pred.low_signal, pred.meta.skill, vector(pid), store.reference, pid)
print(pid, "IDH call:", prediction.call, prediction.verdict.value, "| true:", truth(pid, Marker.IDH))
display(pd.DataFrame([{"source": c.source, "level": c.level, "value": c.value, "detail": c.detail} for c in components]))
print("Recommended action:", primary(components).action)

UCSF-PDGM-0061

IDH call:

wildtype

CONTESTED

| true:

mutant

,source,level,value,detail
0,multiplicity,medium,0.0000,5 of 5 equally good models agree; their probab...
1,ambiguity,low,0.2838,Median probability 0.22 is 0.28 from the decis...
2,data_sparsity,high,1.0000,This tumor is further from its nearest trainin...
3,missing_inputs,low,0.0000,All expected MRI inputs are present.
4,model_skill,low,0.6056,The best model is 61% better than guessing fro...


Recommended action:

Do not rely on the AI here; refer for expert review: the tumor is unlike training cases.

## 6. Held-out performance and the stability check

All 99 test patients, every deployed set: AUC of the set median with a bootstrap 95% CI, and the error rate within
each verdict. If stability means something, Stable calls should be wrong far less often than Contested/Arbitrary ones.
Computed here by `ml/pipeline/evaluate.py` (the same numbers it writes to `artifacts/metrics/heldout.json`).

In [7]:
import evaluate

rows, per_patient=[], {}
for entry in evaluate.yaml.safe_load((ROOT/"ml"/"configs"/"markers.yaml").read_text())["markers"]:
    result, patients=evaluate.evaluate_set(entry["marker"], ROOT/"artifacts"/entry["artifact_dir"], test)
    per_patient[(result["marker"], result["variant"])]=patients
    v=result["by_verdict"]
    rows.append({
        "marker": result["marker"], "variant": result["variant"], "n": result["n_test"],
        "AUC": result["auc"], "AUC 95% CI": result.get("auc_ci95"),
        "Stable n / error": f"{v['STABLE']['n']} / {v['STABLE']['error_rate']}",
        "Contested n / error": f"{v['CONTESTED']['n']} / {v['CONTESTED']['error_rate']}",
        "Arbitrary n / error": f"{v['ARBITRARY']['n']} / {v['ARBITRARY']['error_rate']}",
    })
pd.DataFrame(rows)

,marker,variant,n,AUC,AUC 95% CI,Stable n / error,Contested n / error,Arbitrary n / error
0,IDH,fused,99,0.9772,"[0.9411, 0.9986]",90 / 0.0222,7 / 0.4286,2 / 0.5
1,1p19q,fused,80,0.8974,"[0.7722, 1.0]",80 / 0.025,0 / None,0 / None
2,MGMT,fused,84,0.5525,"[0.3991, 0.7009]",84 / 0.2976,0 / None,0 / None
3,GRADE,fused,99,0.9988,"[0.9943, 1.0]",90 / 0.0111,7 / 0.1429,2 / 1.0
4,IDH,imaging_only,99,0.9633,"[0.925, 0.9888]",93 / 0.0645,5 / 0.6,1 / 1.0
5,1p19q,imaging_only,80,0.9038,"[0.7975, 0.9873]",80 / 0.025,0 / None,0 / None
6,MGMT,imaging_only,84,0.5797,"[0.4262, 0.7197]",84 / 0.2976,0 / None,0 / None
7,GRADE,imaging_only,99,0.9963,"[0.9872, 1.0]",92 / 0.0217,5 / 0.2,2 / 1.0
8,1p19q,fused_given_idh,80,0.9744,"[0.9303, 1.0]",80 / 0.025,0 / None,0 / None
9,MGMT,fused_given_idh,84,0.6258,"[0.48, 0.7559]",84 / 0.2976,0 / None,0 / None


## 7. Resolved by combining

Test patients whose MRI-only set is not Stable but whose MRI + patient-context set is Stable, with the accuracy of
the combined call on them, and the reverse direction (Stable on MRI alone, not once context is added).

In [8]:
pd.DataFrame([
    {"marker": m, **evaluate.resolved_by_combining(per_patient[(m, "fused")], per_patient[(m, "imaging_only")])}
    for m in ("IDH", "1p19q", "MGMT", "GRADE")
])

,marker,n_test,imaging_only_not_stable,resolved_by_combining,fused_accuracy_on_resolved,stable_only_without_context
0,IDH,99,6,3,1.0,6
1,1p19q,80,0,0,NaN,0
2,MGMT,84,0,0,NaN,0
3,GRADE,99,7,2,1.0,4
